In [1]:
from pyspark.sql import SparkSession

# Khởi tạo Spark Session
spark = SparkSession.builder.appName("IoTSensorAnalysis").getOrCreate()

#  Đọc file iot_sensors.csv
df_iot = spark.read.csv("data/iot_sensors.csv", header=True, inferSchema=True)

# Kiểm tra schema
print("\n. Cấu trúc Schema:")
df_iot.printSchema()

# Đếm tổng số dòng dữ liệu
total_rows = df_iot.count()
print(f"\n. Tổng số dòng dữ liệu: {total_rows}")

--- PHẦN A: ĐỌC DỮ LIỆU ---

2. Cấu trúc Schema:
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: integer (nullable = true)
 |-- vibration: double (nullable = true)
 |-- status: string (nullable = true)


3. Tổng số dòng dữ liệu: 4


In [ ]:
# Phần B phân tích thống kê

In [2]:
from pyspark.sql.functions import avg, max, min

# Tính nhiệt độ trung bình, lớn nhất, nhỏ nhất
print("\n. Thống kê nhiệt độ:")
df_iot.select(
    avg("temperature").alias("nhiet_do_TB"),
    max("temperature").alias("nhiet_do_MAX"),
    min("temperature").alias("nhiet_do_MIN")
).show()

# Tính độ rung trung bình theo từng cảm biến
print(" Độ rung trung bình theo từng cảm biến:")
df_iot.groupBy("sensor_id").agg(avg("vibration").alias("do_rung_TB")).show()


. Thống kê nhiệt độ:
+-----------+------------+------------+
|nhiet_do_TB|nhiet_do_MAX|nhiet_do_MIN|
+-----------+------------+------------+
|       48.4|        82.1|        35.5|
+-----------+------------+------------+

 Độ rung trung bình theo từng cảm biến:
+---------+----------+
|sensor_id|do_rung_TB|
+---------+----------+
|     S001|     11.75|
|     S002|      55.0|
|     S003|      10.2|
+---------+----------+



In [ ]:
# Phần C Phát hiện bất thường

In [5]:
from pyspark.sql.functions import col

# Lọc ra các dòng có temperature > 80 hoặc vibration > 50
df_anomaly = df_iot.filter((col("temperature") > 80) | (col("vibration") > 50))
print("\n. Các bản ghi bất thường (Nhiệt độ > 80 hoặc Độ rung > 50):")
df_anomaly.show()

#  Đếm số lượng bản ghi bất thường theo sensor_id
print(" Số lượng bản ghi bất thường theo từng cảm biến:")
df_anomaly.groupBy("sensor_id").count().alias("so_lan_bat_thuong").show()


. Các bản ghi bất thường (Nhiệt độ > 80 hoặc Độ rung > 50):
+---------+-------------------+-----------+--------+---------+-------+
|sensor_id|          timestamp|temperature|humidity|vibration| status|
+---------+-------------------+-----------+--------+---------+-------+
|     S002|2026-04-01 08:01:00|       82.1|      65|     55.0|warning|
+---------+-------------------+-----------+--------+---------+-------+

 Số lượng bản ghi bất thường theo từng cảm biến:
+---------+-----+
|sensor_id|count|
+---------+-----+
|     S002|    1|
+---------+-----+



In [ ]:
# Phần D Phân tích theo thời gian

In [6]:
from pyspark.sql.functions import to_date

# Tạo cột ngày từ timestamp
df_iot_date = df_iot.withColumn("date", to_date(col("timestamp")))
print("\n8.Bảng dữ liệu sau khi thêm cột ngày (date):")
df_iot_date.show(5)

# Tính nhiệt độ trung bình theo ngày
print("Nhiệt độ trung bình theo ngày:")
df_iot_date.groupBy("date").agg(avg("temperature").alias("nhiet_do_TB_ngay")).show()

#  Tính số lượng trạng thái status theo từng loại
print("Thống kê số lượng theo trạng thái (status):")
df_iot_date.groupBy("status").count().show()


8.Bảng dữ liệu sau khi thêm cột ngày (date):
+---------+-------------------+-----------+--------+---------+-------+----------+
|sensor_id|          timestamp|temperature|humidity|vibration| status|      date|
+---------+-------------------+-----------+--------+---------+-------+----------+
|     S001|2026-04-01 08:00:00|       35.5|      70|     12.0| normal|2026-04-01|
|     S002|2026-04-01 08:01:00|       82.1|      65|     55.0|warning|2026-04-01|
|     S001|2026-04-01 08:02:00|       36.0|      69|     11.5| normal|2026-04-01|
|     S003|2026-04-01 08:03:00|       40.0|      72|     10.2| normal|2026-04-01|
+---------+-------------------+-----------+--------+---------+-------+----------+

Nhiệt độ trung bình theo ngày:
+----------+----------------+
|      date|nhiet_do_TB_ngay|
+----------+----------------+
|2026-04-01|            48.4|
+----------+----------------+

Thống kê số lượng theo trạng thái (status):
+-------+-----+
| status|count|
+-------+-----+
| normal|    3|
|warnin

Câu 1: Dữ liệu IoT thể hiện rõ nhất đặc trưng nào của Big Data?
Dữ liệu IoT thể hiện rõ nhất hai đặc trưng: Velocity (Tốc độ) và Volume (Dung lượng).

Velocity: Hàng nghìn, hàng vạn cảm biến trong nhà máy liên tục gửi dữ liệu về hệ thống từng giây, từng mili-giây mà không hề ngừng nghỉ.

Volume: Tốc độ sinh dữ liệu khủng khiếp đó tích tụ lại tạo ra một khối lượng dữ liệu khổng lồ (hàng Terabyte đến Petabyte) chỉ trong một thời gian rất ngắn.

Câu 2: Vì sao dữ liệu cảm biến thường cần xử lý gần thời gian thực (near real-time)?
Vì tính chất "sống còn" của hệ thống phần cứng. Mục đích chính của việc lắp cảm biến là để giám sát tình trạng hoạt động.
Ví dụ: Nếu một cỗ máy đang nóng lên quá mức (quá nhiệt) hoặc rung lắc dữ dội, hệ thống cần phát hiện và cảnh báo ngay lập tức để ngắt điện hoặc bảo trì kịp thời. Nếu đợi gom dữ liệu đến cuối ngày mới chạy xử lý theo lô (batch processing) thì máy móc có thể đã hỏng hóc nặng, làm gián đoạn dây chuyền hoặc thậm chí gây ra cháy nổ rồi.

Câu 3: Ngưỡng bất thường trong bài này được đặt thủ công; trong thực tế có thể thay bằng phương pháp nào thông minh hơn?
Việc đặt ngưỡng cứng (temperature > 80) rất thụ động và không linh hoạt, vì mỗi loại máy móc lại có một dải hoạt động an toàn khác nhau. Trong thực tế, các hệ thống Big Data hiện đại sẽ tích hợp các mô hình Machine Learning (Học máy) và Deep Learning để tự động phát hiện bất thường (Anomaly Detection):

Dựa trên thống kê động: Thuật toán tự động tính toán đường trung bình trượt (Moving Average) và độ lệch chuẩn (Z-score) theo thời gian. Bất kỳ giá trị nào nhảy vọt ra khỏi vùng an toàn này sẽ bị đánh dấu.

Học không giám sát (Unsupervised Learning): Sử dụng các thuật toán như Isolation Forest hoặc DBSCAN để máy tính tự học đâu là trạng thái "bình thường" từ dữ liệu lịch sử, sau đó tự động cô lập các điểm dữ liệu dị biệt (outliers) mà không cần con người mớm trước ngưỡng.
